In [1]:
from pyspark.sql import SparkSession # type: ignore
from pyspark.sql.types import * # type: ignore
from pyspark.sql.functions import * #type: ignore

spark = SparkSession.builder \
    .appName("Test") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

In [2]:
data = [
    (1, "Alice", 30),
    (2, "Bob", None),
    (3, None, 35),
    (4, "David", 28),
    (5, "Eva", None),
    (6, None, None),
    (7, "Grace", 27),
    (8, "Hannah", 32),
    (9, None, 29),
    (10, "Jack", 45),
]
df = spark.createDataFrame(data, ["id", "name", "age"])
df.show()

+---+------+----+
| id|  name| age|
+---+------+----+
|  1| Alice|  30|
|  2|   Bob|NULL|
|  3|  NULL|  35|
|  4| David|  28|
|  5|   Eva|NULL|
|  6|  NULL|NULL|
|  7| Grace|  27|
|  8|Hannah|  32|
|  9|  NULL|  29|
| 10|  Jack|  45|
+---+------+----+



In [3]:
df.select("name","age").show()

+------+----+
|  name| age|
+------+----+
| Alice|  30|
|   Bob|NULL|
|  NULL|  35|
| David|  28|
|   Eva|NULL|
|  NULL|NULL|
| Grace|  27|
|Hannah|  32|
|  NULL|  29|
|  Jack|  45|
+------+----+



In [4]:
df.select("name").limit(5).show()

+-----+
| name|
+-----+
|Alice|
|  Bob|
| NULL|
|David|
|  Eva|
+-----+



In [5]:
df1 = df.withColumnRenamed("name", "full_name") \
       .withColumnRenamed("age", "Full_age")

df.show()

+---+------+----+
| id|  name| age|
+---+------+----+
|  1| Alice|  30|
|  2|   Bob|NULL|
|  3|  NULL|  35|
|  4| David|  28|
|  5|   Eva|NULL|
|  6|  NULL|NULL|
|  7| Grace|  27|
|  8|Hannah|  32|
|  9|  NULL|  29|
| 10|  Jack|  45|
+---+------+----+



In [6]:
df.withColumn("country",lit("USA")).show()

+---+------+----+-------+
| id|  name| age|country|
+---+------+----+-------+
|  1| Alice|  30|    USA|
|  2|   Bob|NULL|    USA|
|  3|  NULL|  35|    USA|
|  4| David|  28|    USA|
|  5|   Eva|NULL|    USA|
|  6|  NULL|NULL|    USA|
|  7| Grace|  27|    USA|
|  8|Hannah|  32|    USA|
|  9|  NULL|  29|    USA|
| 10|  Jack|  45|    USA|
+---+------+----+-------+



In [7]:
from pyspark.sql.functions import col, when

df1 = df.withColumn(
    "country",
    when(col("id") % 2 != 0, "India").otherwise("USA")
)

In [8]:
# df1.select(col("country")=="India").show()

df1.show()

+---+------+----+-------+
| id|  name| age|country|
+---+------+----+-------+
|  1| Alice|  30|  India|
|  2|   Bob|NULL|    USA|
|  3|  NULL|  35|  India|
|  4| David|  28|    USA|
|  5|   Eva|NULL|  India|
|  6|  NULL|NULL|    USA|
|  7| Grace|  27|  India|
|  8|Hannah|  32|    USA|
|  9|  NULL|  29|  India|
| 10|  Jack|  45|    USA|
+---+------+----+-------+



In [9]:
# df1 = df.withColumn(
#     "country",
#     when(col("id") % 2 != 0, "India").otherwise("USA")
# )

df1 = df.withColumn(
    "country",
    when(col("id")%2 !=0 , "India")
    .when(col("id")%3 ==0 , "Paris")
    .otherwise("USA")
)

df1.show()

+---+------+----+-------+
| id|  name| age|country|
+---+------+----+-------+
|  1| Alice|  30|  India|
|  2|   Bob|NULL|    USA|
|  3|  NULL|  35|  India|
|  4| David|  28|    USA|
|  5|   Eva|NULL|  India|
|  6|  NULL|NULL|  Paris|
|  7| Grace|  27|  India|
|  8|Hannah|  32|    USA|
|  9|  NULL|  29|  India|
| 10|  Jack|  45|    USA|
+---+------+----+-------+



In [10]:
df2 = df1.withColumn("demo",lit("mockval"))

df2.show()

+---+------+----+-------+-------+
| id|  name| age|country|   demo|
+---+------+----+-------+-------+
|  1| Alice|  30|  India|mockval|
|  2|   Bob|NULL|    USA|mockval|
|  3|  NULL|  35|  India|mockval|
|  4| David|  28|    USA|mockval|
|  5|   Eva|NULL|  India|mockval|
|  6|  NULL|NULL|  Paris|mockval|
|  7| Grace|  27|  India|mockval|
|  8|Hannah|  32|    USA|mockval|
|  9|  NULL|  29|  India|mockval|
| 10|  Jack|  45|    USA|mockval|
+---+------+----+-------+-------+



In [11]:
df3 = df2.drop("demo")

In [12]:
df3.show()

+---+------+----+-------+
| id|  name| age|country|
+---+------+----+-------+
|  1| Alice|  30|  India|
|  2|   Bob|NULL|    USA|
|  3|  NULL|  35|  India|
|  4| David|  28|    USA|
|  5|   Eva|NULL|  India|
|  6|  NULL|NULL|  Paris|
|  7| Grace|  27|  India|
|  8|Hannah|  32|    USA|
|  9|  NULL|  29|  India|
| 10|  Jack|  45|    USA|
+---+------+----+-------+



In [13]:
# df3.filter(col("age")>20 & col("age") < 30).show()
df3.filter((col("age") > 20) & (col("age") < 30)).show()

+---+-----+---+-------+
| id| name|age|country|
+---+-----+---+-------+
|  4|David| 28|    USA|
|  7|Grace| 27|  India|
|  9| NULL| 29|  India|
+---+-----+---+-------+



In [14]:
df3.filter(col("name").startswith("D")).show()

+---+-----+---+-------+
| id| name|age|country|
+---+-----+---+-------+
|  4|David| 28|    USA|
+---+-----+---+-------+



In [15]:
df.filter(col("name").endswith("e")).show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 30|
|  7|Grace| 27|
+---+-----+---+



In [18]:
df_fil = df.filter(df.name.isNotNull()).show()

+---+------+----+
| id|  name| age|
+---+------+----+
|  1| Alice|  30|
|  2|   Bob|NULL|
|  4| David|  28|
|  5|   Eva|NULL|
|  7| Grace|  27|
|  8|Hannah|  32|
| 10|  Jack|  45|
+---+------+----+

